# 1: Data Ingestion
**Project:** Academic Paper Title Generation  
**Dataset:** `librarian-bots/arxiv-metadata-snapshot`  
**Goal:** Load the dataset, inspect its structure, verify data quality, and compute summary statistics.

---

## Dataset Choice & Project Goal

**Dataset:** [`librarian-bots/arxiv-metadata-snapshot`](https://huggingface.co/datasets/librarian-bots/arxiv-metadata-snapshot)

We chose this dataset for the following reasons:

- **Task alignment:** Contains explicit `title` and `abstract` fields, mapping directly to our abstract → title generation task
- **STEM-exclusive:** arXiv only hosts scientific and technical papers (CS, Physics, Math, Biology, Statistics, etc.)
- **Cross-domain support:** The `categories` field enables cross-domain experiments without needing multiple datasets
- **Recency:** Updated weekly — papers up to March 2026
- **Lightweight:** Metadata only (no full article text), keeping memory usage manageable
- **Traceability:** Each paper has an `id` to retrieve the full paper at `arxiv.org/abs/{id}`
- **Source:** Available on HuggingFace as recommended

**Project Goal:** Train and compare seq2seq architectures to generate concise, accurate titles for STEM academic papers given their abstracts. Evaluate cross-domain generalization across arXiv subject categories using ROUGE, BLEU, and BERTScore.

---

## Imports

In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import os

c:\Users\Widi\anaconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Dataset:

In [2]:
print("Loading dataset from HuggingFace...")
dataset = load_dataset("librarian-bots/arxiv-metadata-snapshot")

#structure:
print(dataset)
print(f"\nColumn names: {dataset['train'].column_names}")

Loading dataset from HuggingFace...


DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 3021763
    })
})

Column names: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']


In [3]:
#keeping only relavent columns
KEEP_COLUMNS = ["id", "title", "abstract", "categories"]

raw = dataset["train"].select_columns(KEEP_COLUMNS)
df = raw.to_pandas()

#removing whitespace from all text fields
df["title"]      = df["title"].str.strip()
df["abstract"]   = df["abstract"].str.strip()
df["categories"] = df["categories"].str.strip()

#checking df
print(f"DataFrame shape: {df.shape}")
df.head()

DataFrame shape: (3021763, 4)


,id,title,abstract,categories
0,1009.3998,An inverse theorem for the Gowers U^{s+1}[N]-norm,We prove the inverse conjecture for the Gowers...,math.CO math.DS
1,1307.4144,The Origin of the Dynamical Quantum Non-locality,Non-locality is one of the hallmarks of quantu...,quant-ph gr-qc hep-th physics.atom-ph physics....
2,1601.05454,Bounding 2D Functions by Products of 1D Functions,"Given sets $X,Y$ and a regular cardinal $\mu$,...",math.LO
3,1805.00656,Rolled-up self-assembly of compact magnetic in...,Three-dimensional self-assembly of lithographi...,physics.app-ph
4,1904.00563,Proper 3-orientations of bipartite planar grap...,We show that every bipartite planar graph with...,math.CO


## Data Summary:

In [4]:
#generic info:
print(f"Dataset ID  : librarian-bots/arxiv-metadata-snapshot")
print(f"Source      : HuggingFace Hub")
print(f"Format      : Apache Parquet (columnar, compressed)")
print(f"Domain      : STEM academic papers (arXiv)")
print(f"Columns kept: {KEEP_COLUMNS}")
print("\n")
#df info:
df.info()
print("\n")
print(f"Total rows: {len(df):,}")
print(f"Columns   : {len(df.columns)}")
print(f"Data types:")
for col, dtype in df.dtypes.items():
    print(f"    {col:<15}: {dtype}")

Dataset ID  : librarian-bots/arxiv-metadata-snapshot
Source      : HuggingFace Hub
Format      : Apache Parquet (columnar, compressed)
Domain      : STEM academic papers (arXiv)
Columns kept: ['id', 'title', 'abstract', 'categories']


<class 'pandas.DataFrame'>
RangeIndex: 3021763 entries, 0 to 3021762
Data columns (total 4 columns):
 #   Column      Dtype
---  ------      -----
 0   id          str  
 1   title       str  
 2   abstract    str  
 3   categories  str  
dtypes: str(4)
memory usage: 3.2 GB


Total rows: 3,021,763
Columns   : 4
Data types:
    id             : str
    title          : str
    abstract       : str
    categories     : str


In [5]:
#stats w/ sample:
sample_10k = df.sample(n=10_000, random_state=42)

title_wc    = sample_10k["title"].str.split().apply(len)
abstract_wc = sample_10k["abstract"].str.split().apply(len)
title_cc    = sample_10k["title"].str.len()
abstract_cc = sample_10k["abstract"].str.len()

stats = pd.DataFrame({
    "Title (words)"    : title_wc.describe(),
    "Abstract (words)" : abstract_wc.describe(),
    "Title (chars)"    : title_cc.describe(),
    "Abstract (chars)" : abstract_cc.describe(),
}).round(1)

print(stats.to_string())


       Title (words)  Abstract (words)  Title (chars)  Abstract (chars)
count        10000.0           10000.0        10000.0           10000.0
mean             9.9             145.7           76.0             999.1
std              3.7              62.2           27.7             429.2
min              1.0               7.0            7.0              40.0
25%              7.0              99.0           56.0             672.0
50%              9.0             143.0           75.0             980.0
75%             12.0             189.0           92.0            1313.0
max             33.0             383.0          220.0            2513.0


## Qual check

In [6]:
quality = pd.DataFrame({
    "null_count"    : df[KEEP_COLUMNS].isnull().sum(),
    "empty_strings" : (df[KEEP_COLUMNS] == "").sum(),
    "unique_values" : df[KEEP_COLUMNS].nunique(),
})

print(quality.to_string())

dupe_count = df.duplicated(subset=["id"]).sum()
print("")
print(f"Dupe IDs : {dupe_count:,}")

            null_count  empty_strings  unique_values
id                   0              0        3021737
title                0              0        3017396
abstract             0              0        3019915
categories           0              0          96908

Dupe IDs : 26


## Category

Since each row can have multiple categories (cs.LG cs.AI etc.), I wanted to get an idea

In [7]:
# Each row can have multiple categories e.g. 'cs.LG cs.AI math.ST'
all_cats = df["categories"].str.split().explode()
top_cats = all_cats.value_counts().head(25)

#Top 25 categories
print(top_cats.to_string())

categories
cs.LG                 262731
hep-ph                195475
cs.CV                 189805
hep-th                181703
quant-ph              178013
cs.AI                 173990
gr-qc                 121058
cond-mat.mtrl-sci     108299
cs.CL                 107394
astro-ph              105380
cond-mat.mes-hall     100793
math-ph                89371
math.MP                89371
cond-mat.str-el        82412
cond-mat.stat-mech     81059
math.CO                77608
stat.ML                77007
astro-ph.GA            76776
astro-ph.CO            76537
math.AP                73451
astro-ph.SR            68972
astro-ph.HE            68383
math.PR                65209
nucl-th                62250
math.OC                60822


In [8]:
domains = {
    "cs"      : "Computer Science",
    "math"    : "Mathematics",
    "physics" : "Physics",
    "quant-ph": "Quantum Physics",
    "stat"    : "Statistics",
    "q-bio"   : "Quantitative Biology",
    "econ"    : "Economics",
    "hep"     : "High Energy Physics",
    "gr-qc"   : "General Relativity",
    "cond-mat": "Condensed Matter",
}

domain_counts = {}
for prefix, label in domains.items():
    count = df["categories"].str.contains(prefix, regex=False).sum()
    domain_counts[label] = count

domain_df = pd.Series(domain_counts).sort_values(ascending=False)
print("\nHigh-level domain distribution:")
for domain, count in domain_df.items():
    pct = count / len(df) * 100
    print(f" {domain:<20}: {count:>5,}  ({pct:.1f}%)")


High-level domain distribution:
 Computer Science    : 1,189,097  (39.4%)
 Mathematics         : 750,032  (24.8%)
 Condensed Matter    : 416,961  (13.8%)
 High Energy Physics : 386,905  (12.8%)
 Physics             : 302,896  (10.0%)
 Statistics          : 224,432  (7.4%)
 Quantum Physics     : 178,013  (5.9%)
 General Relativity  : 121,058  (4.0%)
 Quantitative Biology: 54,660  (1.8%)
 Economics           : 15,295  (0.5%)


## Domain Filtering

I'm going to filter by the primary domain using a regex on the category fields|

In [9]:
#regex=True w/ an escaped dot to avoid partial matches
cs_df      = df[df["categories"].str.contains(r"cs\.",      regex=True)].reset_index(drop=True)
physics_df = df[df["categories"].str.contains(r"physics\.", regex=True)].reset_index(drop=True)
math_df    = df[df["categories"].str.contains(r"math\.",    regex=True)].reset_index(drop=True)
bio_df     = df[df["categories"].str.contains(r"q-bio\.",   regex=True)].reset_index(drop=True)

print("Domain split sizes:")
print(f"CS papers     : {len(cs_df):,}")
print(f"Physics papers: {len(physics_df):,}")
print(f"Math papers   : {len(math_df):,}")
print(f"Bio papers    : {len(bio_df):,}")

Domain split sizes:
CS papers     : 1,189,097
Physics papers: 302,896
Math papers   : 750,032
Bio papers    : 53,307


The plan is to train on CS, evaluate cross-domain on Physics-> Math -> Bio

## Drawing a Working Sample

In [10]:
SAMPLE_SIZE = 50_000
RANDOM_SEED = 42 #rows

df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

#category distribution in sample:
sample_cats = df_sample["categories"].str.split().explode().value_counts().head(10)
print(sample_cats.to_string())

#saved for next notebooks (saved in ext)
os.makedirs("../ext", exist_ok=True)
df_sample.to_parquet("../ext/sample_50k.parquet", index=False)

categories
cs.LG                4354
hep-ph               3186
cs.CV                3160
quant-ph             3026
hep-th               2973
cs.AI                2855
gr-qc                1994
cond-mat.mtrl-sci    1833
cs.CL                1758
astro-ph             1720
